In [1]:
import pandas as pd
df = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/train_data with labels.xlsx')
print(df['label'].value_counts()) # Check 0 vs 1 distribution

label
0    4088
1    4088
Name: count, dtype: int64


In [2]:
def extract_linguistic_features(text):
    words = text.split()
    sentences = text.split('.')
    avg_word_len = sum(len(w) for w in words) / (len(words) + 1e-5)
    avg_sent_len = len(words) / (len(sentences) + 1e-5)
    vocab_richness = len(set(words)) / (len(words) + 1e-5)
    return [avg_word_len, avg_sent_len, vocab_richness]

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize the vectorizer. We limit it to the top 5,000 words
# so Streamlit web app doesn't crash from low memory later.
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')

# 2. Transform text column into a matrix of numbers
X_tfidf = tfidf_vectorizer.fit_transform(df['text'].astype(str))

# Print the shape to see new numerical grid
print("TF-IDF Matrix Shape:", X_tfidf.shape)
# Example output: (Rows in your dataset, 5000 columns)

TF-IDF Matrix Shape: (8176, 5000)


In [4]:
!pip install gensim
import gensim.downloader as api
import numpy as np

print("Downloading pre-trained GloVe embeddings (this takes about 1-2 minutes)...")
# 'glove-wiki-gigaword-100' gives us a 100-dimension vector for each word
glove_model = api.load("glove-wiki-gigaword-100")
print("GloVe model downloaded successfully!")

# Helper function to convert a whole paragraph/text entry into a single vector
def text_to_glove_vector(text):
    if not isinstance(text, str):
        return np.zeros(100)

    words = text.lower().split()
    vectors = [glove_model[word] for word in words if word in glove_model]

    # If none of the words are in the GloVe dictionary, return a row of zeros
    if len(vectors) == 0:
        return np.zeros(100)

    # Take the average vector of all words in the text to represent the sentence
    return np.mean(vectors, axis=0)

# Apply this function to your entire dataset text column
X_glove = np.array(df['text'].apply(text_to_glove_vector).tolist())

print("GloVe Embedding Matrix Shape:", X_glove.shape)
# Output will be: (Rows in your dataset, 100 columns)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 65.2 MB/s eta 0:00:00
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe model downloaded successfully!
GloVe Embedding Matrix Shape: (8176, 100)


In [5]:
from sklearn.model_selection import train_test_split

# Define your target labels (0 for Human, 1 for AI)
y = df['label'].values

# Split the TF-IDF features (80% training, 20% testing)
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# Split the GloVe features (80% training, 20% testing)
X_train_glove, X_test_glove, _, _ = train_test_split(
    X_glove, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train_tfidf.shape[0]} rows")
print(f"Testing set size: {X_test_tfidf.shape[0]} rows")

Training set size: 6540 rows
Testing set size: 1636 rows


In [6]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import joblib
import os

print("Starting Fast SVM Hyperparameter Tuning...")

# 1. Initialize the model framework
svm_pipeline = SVC(probability=True, random_state=42)

# 2. Narrow down to a fast linear search (excellent for high-dimensional TF-IDF matrices)
param_grid = {
    'C': [0.1, 1.0],
    'kernel': ['linear']
}

# 3. Set up the Grid Search with fast cross-validation
grid_search_svm = GridSearchCV(svm_pipeline, param_grid, cv=2, scoring='accuracy', n_jobs=-1)
grid_search_svm.fit(X_train_tfidf, y_train)

# 4. Extract the best model configurations
best_svm_model = grid_search_svm.best_estimator_
print("Best Parameters Found:", grid_search_svm.best_params_)

# 5. Evaluate predictions on the test set
svm_predictions = best_svm_model.predict(X_test_tfidf)
print("\nSVM Classification Report:")
print(classification_report(y_test, svm_predictions))

# 6. Save the trained model and vectorizer
os.makedirs('models', exist_ok=True)
joblib.dump(best_svm_model, 'models/svm_model.pkl')
joblib.dump(tfidf_vectorizer, 'models/tfidf_vectorizer.pkl')
print("Saved svm_model.pkl and tfidf_vectorizer.pkl to 'models/' directory!")

Starting Fast SVM Hyperparameter Tuning...
Best Parameters Found: {'C': 1.0, 'kernel': 'linear'}

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.96      0.96       818
           1       0.96      0.97      0.97       818

    accuracy                           0.97      1636
   macro avg       0.97      0.97      0.97      1636
weighted avg       0.97      0.97      0.97      1636

Saved svm_model.pkl and tfidf_vectorizer.pkl to 'models/' directory!


In [7]:
from sklearn.tree import DecisionTreeClassifier

print("Starting Decision Tree Hyperparameter Tuning...")

# 1. Initialize the Decision Tree model
dt_pipeline = DecisionTreeClassifier(random_state=42)

# 2. Define hyperparameters to tune
dt_param_grid = {
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# 3. Grid Search with 2-fold cross-validation
grid_search_dt = GridSearchCV(dt_pipeline, dt_param_grid, cv=2, scoring='accuracy', n_jobs=-1)
grid_search_dt.fit(X_train_tfidf, y_train)

# 4. Extract best configuration
best_dt_model = grid_search_dt.best_estimator_
print("Best Decision Tree Parameters Found:", grid_search_dt.best_params_)

# 5. Evaluate on the test set
dt_predictions = best_dt_model.predict(X_test_tfidf)
print("\nDecision Tree Classification Report:")
print(classification_report(y_test, dt_predictions))

# 6. Save the model artifact
joblib.dump(best_dt_model, 'models/decision_tree_model.pkl')
print("Saved decision_tree_model.pkl to 'models/' directory! [cite: 75]")

Starting Decision Tree Hyperparameter Tuning...
Best Decision Tree Parameters Found: {'max_depth': 20, 'min_samples_split': 2}

Decision Tree Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.90      0.91       818
           1       0.90      0.92      0.91       818

    accuracy                           0.91      1636
   macro avg       0.91      0.91      0.91      1636
weighted avg       0.91      0.91      0.91      1636

Saved decision_tree_model.pkl to 'models/' directory! [cite: 75]


In [8]:
from sklearn.ensemble import AdaBoostClassifier

print("Starting AdaBoost Hyperparameter Tuning...")

# 1. Initialize AdaBoost framework
ada_pipeline = AdaBoostClassifier(random_state=42)

# 2. Define parameters to tune
ada_param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.1, 1.0]
}

# 3. Grid Search cross-validation
grid_search_ada = GridSearchCV(ada_pipeline, ada_param_grid, cv=2, scoring='accuracy', n_jobs=-1)
grid_search_ada.fit(X_train_tfidf, y_train)

# 4. Extract best configuration
best_ada_model = grid_search_ada.best_estimator_
print("Best AdaBoost Parameters Found:", grid_search_ada.best_params_)

# 5. Evaluate on the test set
ada_predictions = best_ada_model.predict(X_test_tfidf)
print("\nAdaBoost Classification Report:")
print(classification_report(y_test, ada_predictions))

# 6. Save the model artifact
joblib.dump(best_ada_model, 'models/adaboost_model.pkl')
print("Saved adaboost_model.pkl to 'models/' directory! [cite: 75]")

Starting AdaBoost Hyperparameter Tuning...
Best AdaBoost Parameters Found: {'learning_rate': 1.0, 'n_estimators': 100}

AdaBoost Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.93      0.93       818
           1       0.93      0.93      0.93       818

    accuracy                           0.93      1636
   macro avg       0.93      0.93      0.93      1636
weighted avg       0.93      0.93      0.93      1636

Saved adaboost_model.pkl to 'models/' directory! [cite: 75]


In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Preparing text sequences for LSTM and CNN...")

# 1. Define maximum vocabulary size and sequence length
MAX_VOCAB_SIZE = 20000
MAX_SEQUENCE_LENGTH = 200  # Restricts each text to 200 words max

# 2. Initialize and fit the Tokenizer
dl_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
dl_tokenizer.fit_on_texts(df['text'].astype(str))

# 3. Convert text to integer sequences
X_sequences = dl_tokenizer.texts_to_sequences(df['text'].astype(str))

# 4. Pad sequences to ensure uniform input shape
X_padded = pad_sequences(X_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post')

# 5. Train/Test split for the sequential data (using identical random state)
X_train_pad, X_test_pad, _, _ = train_test_split(
    X_padded, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Save the tokenizer so your Streamlit app can process raw inputs later
joblib.dump(dl_tokenizer, 'models/dl_tokenizer.pkl')

print("Padded Sequence Shape:", X_padded.shape)
print("Saved dl_tokenizer.pkl to 'models/' directory!")

Preparing text sequences for LSTM and CNN...
Padded Sequence Shape: (8176, 200)
Saved dl_tokenizer.pkl to 'models/' directory!


In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.metrics import classification_report

print("Building and training Feedforward Neural Network (FNN)...")

# 1. Define FNN Architecture
fnn_model = Sequential([
    # Input layer matches your 5,000 TF-IDF features
    Dense(128, activation='relu', input_shape=(X_train_tfidf.shape[1],)),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Binary classification output (0 or 1)
])

# 2. Compile Model
fnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Convert sparse matrices to float32 arrays so TensorFlow can process them smoothly
X_train_fnn = X_train_tfidf.toarray()
X_test_fnn = X_test_tfidf.toarray()

# 3. Train the model
history_fnn = fnn_model.fit(
    X_train_fnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# 4. Evaluate FNN
fnn_preds = (fnn_model.predict(X_test_fnn) > 0.5).astype("int32")
print("\nFNN Classification Report:")
print(classification_report(y_test, fnn_preds))

# 5. Save the model in the required format
fnn_model.save('models/fnn_model.h5')
print("Saved fnn_model.h5 to 'models/' directory! [cite: 75]")

Building and training Feedforward Neural Network (FNN)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.8561 - loss: 0.3848 - val_accuracy: 0.9648 - val_loss: 0.1087
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9837 - loss: 0.0564 - val_accuracy: 0.9664 - val_loss: 0.0869
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.9975 - loss: 0.0139 - val_accuracy: 0.9648 - val_loss: 0.1002
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9993 - loss: 0.0069 - val_accuracy: 0.9618 - val_loss: 0.0996
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9998 - loss: 0.0025 - val_accuracy: 0.9633 - val_loss: 0.1093
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step



FNN Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       818
           1       0.98      0.97      0.97       818

    accuracy                           0.97      1636
   macro avg       0.97      0.97      0.97      1636
weighted avg       0.97      0.97      0.97      1636

Saved fnn_model.h5 to 'models/' directory! [cite: 75]


In [11]:
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

print("Building and training Long Short-Term Memory (LSTM) Network...")

# 1. Define LSTM Architecture
lstm_model = Sequential([
    # Embedding layer transforms word integer tokens into 128-dimensional dense vectors
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQUENCE_LENGTH),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2), # 64 LSTM units to capture memory loops
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid') # Final binary prediction output
])

# 2. Compile Model
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Train the Model
history_lstm = lstm_model.fit(
    X_train_pad, y_train,
    epochs=3, # Sequential models train slower, 3 epochs is sufficient on T4 GPU
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# 4. Evaluate LSTM
lstm_preds = (lstm_model.predict(X_test_pad) > 0.5).astype("int32")
print("\nLSTM Classification Report:")
print(classification_report(y_test, lstm_preds))

# 5. Save Model
lstm_model.save('models/lstm_model.h5')
print("Saved lstm_model.h5 to 'models/' directory!")

Building and training Long Short-Term Memory (LSTM) Network...
Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


92/92 ━━━━━━━━━━━━━━━━━━━━ 44s 426ms/step - accuracy: 0.6442 - loss: 0.6226 - val_accuracy: 0.8287 - val_loss: 0.4920
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 36s 389ms/step - accuracy: 0.7985 - loss: 0.4896 - val_accuracy: 0.8028 - val_loss: 0.4503
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 43s 412ms/step - accuracy: 0.8651 - loss: 0.3728 - val_accuracy: 0.8593 - val_loss: 0.3795
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step



LSTM Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.93      0.88       818
           1       0.92      0.80      0.86       818

    accuracy                           0.87      1636
   macro avg       0.87      0.87      0.87      1636
weighted avg       0.87      0.87      0.87      1636

Saved lstm_model.h5 to 'models/' directory!


In [12]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D

print("Building and training 1D CNN for Text Classification...")

# 1. Define CNN Architecture
cnn_model = Sequential([
    Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, input_length=MAX_SEQUENCE_LENGTH),
    Conv1D(filters=128, kernel_size=5, activation='relu'), # Scans windows of 5 consecutive words
    GlobalMaxPooling1D(), # Extracts the most prominent feature maps
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

# 2. Compile Model
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Train the Model
history_cnn = cnn_model.fit(
    X_train_pad, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# 4. Evaluate CNN
cnn_preds = (cnn_model.predict(X_test_pad) > 0.5).astype("int32")
print("\nCNN Classification Report:")
print(classification_report(y_test, cnn_preds))

# 5. Save Model
cnn_model.save('models/cnn_model.h5')
print("Saved cnn_model.h5 to 'models/' directory!")

Building and training 1D CNN for Text Classification...
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 223ms/step - accuracy: 0.6906 - loss: 0.5626 - val_accuracy: 0.8746 - val_loss: 0.3061
Epoch 2/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 240ms/step - accuracy: 0.9281 - loss: 0.1922 - val_accuracy: 0.9388 - val_loss: 0.1395
Epoch 3/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 219ms/step - accuracy: 0.9925 - loss: 0.0417 - val_accuracy: 0.9526 - val_loss: 0.1170
Epoch 4/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 20s 217ms/step - accuracy: 0.9995 - loss: 0.0087 - val_accuracy: 0.9557 - val_loss: 0.1228
Epoch 5/5
92/92 ━━━━━━━━━━━━━━━━━━━━ 23s 248ms/step - accuracy: 0.9995 - loss: 0.0050 - val_accuracy: 0.9541 - val_loss: 0.1264
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step

CNN Classification Report:


              precision    recall  f1-score   support

           0       0.94      0.96      0.95       818
           1       0.96      0.94      0.95       818

    accuracy                           0.95      1636
   macro avg       0.95      0.95      0.95      1636
weighted avg       0.95      0.95      0.95      1636

Saved cnn_model.h5 to 'models/' directory!
